# Kaggle Playground Series S6E8
## Part 1 - Setup & Memory Optimization

# Part 2 - Exploratory Data Analysis (EDA)
Run Part 1 first or load the datasets again.

# Part 3 - Advanced Feature Engineering
Run Part 1 first so `train` and `test` are loaded.

# Part 4 - Competition CatBoost Pipeline

# Part 5 - LightGBM Competition Pipeline

# Part 6 - XGBoost Competition Pipeline

# Part 7 - Optuna Hyperparameter Tuning
Tune CatBoost, LightGBM, and XGBoost using Optuna.

# Part 8 - Ensemble Training & Blending
Train tuned models, create OOF predictions, optimize blend weights, and generate the final submission.

# Part 9 - Model Diagnostics, SHAP & Calibration

In [2]:
# !pip install shap -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.metrics import (
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    brier_score_loss
)
from sklearn.calibration import calibration_curve


ImportError: Numba needs NumPy 2.4 or less. Got NumPy 2.5.

## Load OOF Predictions

In [ ]:
# Expected from previous notebooks:
# oof_cat, oof_lgb, oof_xgb, y

oof = pd.DataFrame({
    "CatBoost": oof_cat,
    "LightGBM": oof_lgb,
    "XGBoost": oof_xgb,
    "Target": y
})

display(oof.head())


## Compare Individual Models

In [ ]:
for c in ["CatBoost","LightGBM","XGBoost"]:
    print(c, roc_auc_score(y,oof[c]))


## ROC Curves

In [ ]:
plt.figure(figsize=(7,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    RocCurveDisplay.from_predictions(y,oof[c],name=c,ax=plt.gca())

plt.title("ROC Curves")
plt.show()


## Precision-Recall Curves

In [ ]:
plt.figure(figsize=(7,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    PrecisionRecallDisplay.from_predictions(
        y,oof[c],name=c,ax=plt.gca()
    )

plt.title("Precision-Recall Curves")
plt.show()


## Calibration

In [ ]:
plt.figure(figsize=(6,6))

for c in ["CatBoost","LightGBM","XGBoost"]:
    prob_true, prob_pred = calibration_curve(y,oof[c],n_bins=10)
    plt.plot(prob_pred,prob_true,label=c)

plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("Calibration")
plt.show()

for c in ["CatBoost","LightGBM","XGBoost"]:
    print(c,"Brier:",brier_score_loss(y,oof[c]))


## SHAP (CatBoost Example)

In [ ]:
# model should be final trained CatBoost model
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X.iloc[:5000])

shap.summary_plot(
    shap_values,
    X.iloc[:5000],
    max_display=20
)


## Prediction Distribution

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(oof_cat,bins=50,alpha=.5,label='CatBoost')
plt.hist(oof_lgb,bins=50,alpha=.5,label='LightGBM')
plt.hist(oof_xgb,bins=50,alpha=.5,label='XGBoost')
plt.legend()
plt.title("Prediction Distribution")
plt.show()


## Error Analysis

In [ ]:
ensemble=(oof_cat+oof_lgb+oof_xgb)/3

errors=pd.DataFrame({
    "target":y,
    "prediction":ensemble
})

errors["abs_error"]=abs(errors.target-errors.prediction)

display(
    errors.sort_values("abs_error",ascending=False).head(20)
)


## Save OOF Predictions

In [ ]:
oof.to_csv("oof_predictions.csv",index=False)
print("Saved oof_predictions.csv")


## Final Checklist

- Compare individual model ROC-AUC
- Inspect SHAP importance
- Check calibration
- Analyze worst predictions
- Keep OOF predictions for stacking

The final notebook can use these diagnostics to refine features, tune models further, or build a stacked meta-model.


In [ ]:
import json
import numpy as np
import pandas as pd

from itertools import product
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

TARGET='addicted_label'
ID='id'

with open('best_params.json') as f:
    BEST=json.load(f)

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)


## Prepare datasets

In [ ]:
cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

# CatBoost
X_cat=X.copy()
X_test_cat=X_test.copy()
cat_idx=[X_cat.columns.get_loc(c) for c in cat_cols]

# LightGBM
X_lgb=X.copy()
X_test_lgb=X_test.copy()
for c in cat_cols:
    X_lgb[c]=X_lgb[c].astype('category')
    X_test_lgb[c]=X_test_lgb[c].astype('category')

# XGBoost
X_xgb=X.copy()
X_test_xgb=X_test.copy()
enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
if len(cat_cols):
    X_xgb[cat_cols]=enc.fit_transform(X_xgb[cat_cols].astype(str))
    X_test_xgb[cat_cols]=enc.transform(X_test_xgb[cat_cols].astype(str))


## Cross-validation

In [ ]:
oof_cat=np.zeros(len(X))
oof_lgb=np.zeros(len(X))
oof_xgb=np.zeros(len(X))

pred_cat=np.zeros(len(X_test))
pred_lgb=np.zeros(len(X_test))
pred_xgb=np.zeros(len(X_test))

for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print(f"Fold {fold}")

    cat=CatBoostClassifier(
        **BEST['catboost'],
        iterations=3000,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=False,
        random_seed=42
    )
    cat.fit(X_cat.iloc[tr],y.iloc[tr],
            cat_features=cat_idx,
            eval_set=(X_cat.iloc[va],y.iloc[va]),
            use_best_model=True)

    oof_cat[va]=cat.predict_proba(X_cat.iloc[va])[:,1]
    pred_cat+=cat.predict_proba(X_test_cat)[:,1]/5

    lgb=LGBMClassifier(
        **BEST['lightgbm'],
        objective='binary',
        metric='auc',
        n_estimators=3000,
        random_state=42
    )
    lgb.fit(X_lgb.iloc[tr],y.iloc[tr],categorical_feature=cat_cols)
    oof_lgb[va]=lgb.predict_proba(X_lgb.iloc[va])[:,1]
    pred_lgb+=lgb.predict_proba(X_test_lgb)[:,1]/5

    xgb=XGBClassifier(
        **BEST['xgboost'],
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        n_estimators=3000,
        random_state=42
    )
    xgb.fit(X_xgb.iloc[tr],y.iloc[tr],verbose=False)
    oof_xgb[va]=xgb.predict_proba(X_xgb.iloc[va])[:,1]
    pred_xgb+=xgb.predict_proba(X_test_xgb)[:,1]/5


## Optimize Blend Weights

In [ ]:
best_auc=0
best_w=None

for wc,wl in product(np.arange(0,1.05,0.05),repeat=2):
    wx=1-wc-wl
    if wx<0:
        continue
    blend=wc*oof_cat+wl*oof_lgb+wx*oof_xgb
    auc=roc_auc_score(y,blend)
    if auc>best_auc:
        best_auc=auc
        best_w=(wc,wl,wx)

print("Best AUC:",best_auc)
print("Weights:",best_w)


## Final Submission

In [ ]:
wc,wl,wx=best_w
final_pred=wc*pred_cat+wl*pred_lgb+wx*pred_xgb

submission=sample.copy()
submission[TARGET]=final_pred
submission.to_csv("submission_ensemble.csv",index=False)

display(submission.head())
print("Saved submission_ensemble.csv")


### Next
Part 9 will add SHAP explainability, error analysis, calibration, threshold experiments, and final diagnostics.

In [ ]:
# !pip install optuna -q

import optuna
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]
X=train_fe[FEATURES]
y=train_fe[TARGET]

cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)


## CatBoost Objective

In [ ]:
from catboost import CatBoostClassifier

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()
cat_idx=[X.columns.get_loc(c) for c in cat_cols]

def objective_cat(trial):

    params={
        'loss_function':'Logloss',
        'eval_metric':'AUC',
        'iterations':2000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
        'depth':trial.suggest_int('depth',4,10),
        'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',1,20),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'verbose':False,
        'random_seed':42
    }

    scores=[]

    for tr,va in cv.split(X,y):

        model=CatBoostClassifier(**params)

        model.fit(
            X.iloc[tr],y.iloc[tr],
            cat_features=cat_idx,
            eval_set=(X.iloc[va],y.iloc[va]),
            early_stopping_rounds=200,
            use_best_model=True
        )

        pred=model.predict_proba(X.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_cat=optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat,n_trials=30)

print(study_cat.best_value)
print(study_cat.best_params)


## LightGBM Objective

In [ ]:
from lightgbm import LGBMClassifier

X_lgb=X.copy()
cat_cols=X_lgb.select_dtypes(include=['object','category']).columns.tolist()
for c in cat_cols:
    X_lgb[c]=X_lgb[c].astype('category')

def objective_lgb(trial):

    params={
        'objective':'binary',
        'metric':'auc',
        'n_estimators':3000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.15,log=True),
        'num_leaves':trial.suggest_int('num_leaves',31,255),
        'min_child_samples':trial.suggest_int('min_child_samples',20,150),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
        'random_state':42
    }

    scores=[]

    for tr,va in cv.split(X_lgb,y):
        model=LGBMClassifier(**params)
        model.fit(X_lgb.iloc[tr],y.iloc[tr],categorical_feature=cat_cols)
        pred=model.predict_proba(X_lgb.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_lgb=optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb,n_trials=30)

print(study_lgb.best_value)
print(study_lgb.best_params)


## XGBoost Objective

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder

X_xgb=X.copy()
cat_cols=X_xgb.select_dtypes(include=['object','category']).columns.tolist()

enc=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)

if len(cat_cols):
    X_xgb[cat_cols]=enc.fit_transform(X_xgb[cat_cols].astype(str))

def objective_xgb(trial):

    params={
        'objective':'binary:logistic',
        'eval_metric':'auc',
        'tree_method':'hist',
        'n_estimators':3000,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.15,log=True),
        'max_depth':trial.suggest_int('max_depth',4,10),
        'min_child_weight':trial.suggest_int('min_child_weight',1,10),
        'subsample':trial.suggest_float('subsample',0.6,1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
        'gamma':trial.suggest_float('gamma',0,5),
        'random_state':42,
        'n_jobs':-1
    }

    scores=[]

    for tr,va in cv.split(X_xgb,y):
        model=XGBClassifier(**params)
        model.fit(X_xgb.iloc[tr],y.iloc[tr],verbose=False)
        pred=model.predict_proba(X_xgb.iloc[va])[:,1]
        scores.append(roc_auc_score(y.iloc[va],pred))

    return np.mean(scores)

study_xgb=optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb,n_trials=30)

print(study_xgb.best_value)
print(study_xgb.best_params)


## Save Best Parameters

In [ ]:
best_params={
    'catboost':study_cat.best_params,
    'lightgbm':study_lgb.best_params,
    'xgboost':study_xgb.best_params
}

import json

with open('best_params.json','w') as f:
    json.dump(best_params,f,indent=4)

print(best_params)
print('Saved best_params.json')


### Next
Part 8 will train all three models using the tuned parameters, generate OOF predictions, and create a weighted ensemble.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

# XGBoost requires numeric input
enc=OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

if len(cat_cols)>0:
    X[cat_cols]=enc.fit_transform(X[cat_cols].astype(str))
    X_test[cat_cols]=enc.transform(X_test[cat_cols].astype(str))


## Parameters

In [ ]:
params=dict(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=7,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=2.0,
    gamma=0.1,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
pred=np.zeros(len(X_test))
scores=[]
fi=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=XGBClassifier(**params)

    model.fit(
        X.iloc[tr],y.iloc[tr],
        eval_set=[(X.iloc[va],y.iloc[va])],
        verbose=200
    )

    val=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val

    auc=roc_auc_score(y.iloc[va],val)
    scores.append(auc)

    pred += model.predict_proba(X_test)[:,1]/cv.n_splits

    fi[f'fold_{fold}']=model.feature_importances_

    print(f"Fold {fold} AUC: {auc:.6f}")

print("\nMean CV:",np.mean(scores))
print("Overall OOF:",roc_auc_score(y,oof))


## Feature Importance

In [ ]:
fi["mean"]=fi.mean(axis=1)
fi=fi.sort_values("mean",ascending=False)

display(fi.head(20))

plt.figure(figsize=(8,8))
fi["mean"].head(20).sort_values().plot.barh()
plt.title("Top 20 XGBoost Features")
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=pred
submission.to_csv("submission_xgboost.csv",index=False)

display(submission.head())
print("Saved submission_xgboost.csv")


### Next

Part 7 will focus on **Optuna hyperparameter tuning** for CatBoost, LightGBM, and XGBoost, saving the best parameters for the final ensemble.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

for c in cat_cols:
    X[c]=X[c].astype('category')
    X_test[c]=X_test[c].astype('category')


## Parameters

In [ ]:
params=dict(
    objective='binary',
    metric='auc',
    n_estimators=5000,
    learning_rate=0.02,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
pred=np.zeros(len(X_test))
scores=[]
fi=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=LGBMClassifier(**params)

    model.fit(
        X.iloc[tr],y.iloc[tr],
        eval_set=[(X.iloc[va],y.iloc[va])],
        eval_metric='auc',
        categorical_feature=cat_cols,
        callbacks=[]
    )

    val=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val

    auc=roc_auc_score(y.iloc[va],val)
    scores.append(auc)

    pred+=model.predict_proba(X_test)[:,1]/cv.n_splits

    fi[f'fold_{fold}']=model.feature_importances_

    print(f"Fold {fold} AUC: {auc:.6f}")

print("\nMean CV:",np.mean(scores))
print("Overall:",roc_auc_score(y,oof))


## Feature Importance

In [ ]:
fi['mean']=fi.mean(axis=1)
fi=fi.sort_values('mean',ascending=False)

display(fi.head(20))

plt.figure(figsize=(8,8))
fi['mean'].head(20).sort_values().plot.barh()
plt.title("Top 20 LightGBM Features")
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=pred
submission.to_csv("submission_lightgbm.csv",index=False)

display(submission.head())
print("Saved submission_lightgbm.csv")


### Next

Part 6 will include:
- XGBoost with tuned parameters
- OOF prediction saving
- Preparation for weighted blending with CatBoost and LightGBM.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

TARGET='addicted_label'
ID='id'

# Expects train_fe/test_fe from Part 3
FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES]
y=train_fe[TARGET]
X_test=test_fe[FEATURES]

CAT_FEATURES=X.select_dtypes(include=['object','category']).columns.tolist()
cat_idx=[X.columns.get_loc(c) for c in CAT_FEATURES]


## Training Configuration

In [ ]:
params={
    'loss_function':'Logloss',
    'eval_metric':'AUC',
    'iterations':5000,
    'learning_rate':0.02,
    'depth':8,
    'l2_leaf_reg':5,
    'subsample':0.8,
    'random_seed':42,
    'verbose':250
}

cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof=np.zeros(len(X))
test_pred=np.zeros(len(X_test))
scores=[]
feature_importance=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=CatBoostClassifier(**params)

    model.fit(
        X.iloc[tr],
        y.iloc[tr],
        cat_features=cat_idx,
        eval_set=(X.iloc[va],y.iloc[va]),
        use_best_model=True,
        early_stopping_rounds=300
    )

    val_pred=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val_pred

    fold_auc=roc_auc_score(y.iloc[va],val_pred)
    scores.append(fold_auc)

    print("Fold AUC:",round(fold_auc,6))

    test_pred+=model.predict_proba(X_test)[:,1]/cv.n_splits

    feature_importance[f'fold_{fold}']=model.get_feature_importance()

print("\nMean CV:",np.mean(scores))
print("Std CV :",np.std(scores))


## Overall ROC-AUC

In [ ]:
overall=roc_auc_score(y,oof)
print("Overall OOF ROC-AUC:",overall)


## Feature Importance

In [ ]:
feature_importance['mean']=feature_importance.mean(axis=1)
feature_importance=feature_importance.sort_values('mean',ascending=False)

display(feature_importance.head(20))

plt.figure(figsize=(8,8))
feature_importance['mean'].head(20).sort_values().plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=test_pred
submission.to_csv('submission_catboost.csv',index=False)

display(submission.head())

print('Saved: submission_catboost.csv')


## Notes

- This is a strong CatBoost baseline.
- Save the OOF predictions for later blending with LightGBM and XGBoost.
- In Part 5 we'll build a tuned LightGBM pipeline and prepare for ensembling.


In [ ]:
import numpy as np
import pandas as pd

TARGET='addicted_label'
ID='id'

train_fe=train.copy()
test_fe=test.copy()


## Helper Function

In [ ]:
def create_features(df):

    eps=1e-3

    # Total digital activity
    df["total_usage"] = (
        df["social_media_hours"] +
        df["gaming_hours"] +
        df["work_study_hours"]
    )

    # Ratios
    df["screen_sleep_ratio"] = df["daily_screen_time_hours"]/(df["sleep_hours"]+eps)
    df["social_ratio"] = df["social_media_hours"]/(df["daily_screen_time_hours"]+eps)
    df["gaming_ratio"] = df["gaming_hours"]/(df["daily_screen_time_hours"]+eps)
    df["work_ratio"] = df["work_study_hours"]/(df["daily_screen_time_hours"]+eps)

    df["notifications_per_hour"] = df["notifications_per_day"]/(df["daily_screen_time_hours"]+eps)
    df["opens_per_hour"] = df["app_opens_per_day"]/(df["daily_screen_time_hours"]+eps)

    df["weekend_extra"] = df["weekend_screen_time"]-df["daily_screen_time_hours"]

    df["usage_per_age"] = df["daily_screen_time_hours"]/(df["age"]+eps)

    df["stress_usage"] = (
        df["daily_screen_time_hours"] *
        df["stress_level"].map({
            "Low":1,
            "Medium":2,
            "High":3
        }).fillna(2)
    )

    df["sleep_deficit"]=(8-df["sleep_hours"]).clip(lower=0)

    df["high_screen_flag"]=(df["daily_screen_time_hours"]>8).astype(int)
    df["poor_sleep_flag"]=(df["sleep_hours"]<6).astype(int)
    df["high_notification_flag"]=(df["notifications_per_day"]>150).astype(int)

    df["age_bucket"]=pd.cut(
        df["age"],
        bins=[0,18,25,35,50,100],
        labels=["Teen","18-25","26-35","36-50","50+"]
    )

    return df


In [ ]:
train_fe=create_features(train_fe)
test_fe=create_features(test_fe)

print("Original Features :", train.shape[1]-2)
print("Engineered Features :", train_fe.shape[1]-2)
display(train_fe.head())


## Missing Value Handling

In [ ]:
FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

CAT_FEATURES=train_fe[FEATURES].select_dtypes(
    include=["object","category"]
).columns.tolist()

NUM_FEATURES=[c for c in FEATURES if c not in CAT_FEATURES]

for c in CAT_FEATURES:
    train_fe[c]=train_fe[c].fillna("Missing")
    test_fe[c]=test_fe[c].fillna("Missing")

for c in NUM_FEATURES:
    med=train_fe[c].median()
    train_fe[c]=train_fe[c].fillna(med)
    test_fe[c]=test_fe[c].fillna(med)

print("Categorical:",len(CAT_FEATURES))
print("Numerical :",len(NUM_FEATURES))


## Quick CV Check

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

X=train_fe.drop(columns=[ID,TARGET])
y=train_fe[TARGET]
X_test=test_fe.drop(columns=[ID])

cat_idx=[X.columns.get_loc(c) for c in CAT_FEATURES]

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores=[]

for fold,(tr,va) in enumerate(cv.split(X,y),1):

    model=CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=False,
        random_seed=42
    )

    model.fit(
        X.iloc[tr],
        y.iloc[tr],
        cat_features=cat_idx,
        eval_set=(X.iloc[va],y.iloc[va]),
        use_best_model=True
    )

    pred=model.predict_proba(X.iloc[va])[:,1]
    auc=roc_auc_score(y.iloc[va],pred)
    scores.append(auc)

    print(f"Fold {fold}: {auc:.6f}")

print("\nMean CV:",np.mean(scores))


## Next
In Part 4 we'll build a fully optimized CatBoost pipeline with:
- Out-of-fold predictions
- Test prediction averaging
- Feature importance
- Submission generation
- Tunable parameters


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

TARGET='addicted_label'

# Uncomment if needed
# train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')


## Dataset Shape

In [ ]:
print("Rows :", train.shape[0])
print("Columns :", train.shape[1])

display(train.head())


## Target Distribution

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x=TARGET,data=train)
plt.title("Target Distribution")
plt.show()

print(train[TARGET].value_counts(normalize=True))


## Missing Values

In [ ]:
missing=train.isnull().sum().sort_values(ascending=False)
missing=missing[missing>0]

display(missing.to_frame("Missing"))

if len(missing)>0:
    plt.figure(figsize=(8,max(3,len(missing)*0.4)))
    missing.plot(kind='barh')
    plt.title("Missing Values")
    plt.show()


## Numerical Feature Distributions

In [ ]:
num_cols=train.select_dtypes(include=['number']).columns.tolist()
num_cols=[c for c in num_cols if c!=TARGET and c!='id']

for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.histplot(train[col],kde=True,bins=30)
    plt.title(col)
    plt.show()


## Numerical Features vs Target

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(data=train,x=TARGET,y=col)
    plt.title(col)
    plt.show()


## Categorical Features

In [ ]:
cat_cols=train.select_dtypes(include=['object','category']).columns.tolist()

for col in cat_cols:
    plt.figure(figsize=(7,4))
    sns.countplot(data=train,x=col,hue=TARGET)
    plt.xticks(rotation=30)
    plt.title(col)
    plt.tight_layout()
    plt.show()


## Correlation Heatmap

In [ ]:
corr=train[num_cols+[TARGET]].corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr,cmap='coolwarm',center=0)
plt.title("Correlation Matrix")
plt.show()


## Outlier Detection

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,2.5))
    sns.boxplot(x=train[col])
    plt.title(col)
    plt.show()


## Quick Feature Importance (CatBoost)

In [ ]:
from catboost import CatBoostClassifier

X=train.drop(columns=['id',TARGET])
y=train[TARGET]

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

for col in cat_cols:
    X[col]=X[col].fillna('Missing')

for col in X.columns:
    if col not in cat_cols:
        X[col]=X[col].fillna(X[col].median())

cat_idx=[X.columns.get_loc(c) for c in cat_cols]

model=CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    verbose=False,
    random_seed=42
)

model.fit(X,y,cat_features=cat_idx)

fi=pd.DataFrame({
    'Feature':X.columns,
    'Importance':model.get_feature_importance()
}).sort_values('Importance',ascending=False)

display(fi)

plt.figure(figsize=(8,6))
sns.barplot(data=fi.head(15),x='Importance',y='Feature')
plt.title("Top 15 Features")
plt.show()


### Observations

Fill this section with insights before moving to feature engineering.

In [ ]:
# ============================================================
# Libraries
# ============================================================
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.style.use("ggplot")


In [ ]:
# ============================================================
# Configuration
# ============================================================

SEED = 42
TARGET = "addicted_label"
ID = "id"
N_SPLITS = 5

random.seed(SEED)
np.random.seed(SEED)


## Memory Reduction Utility

In [ ]:
def reduce_memory(df):
    start_mem = df.memory_usage(deep=True).sum()/1024**2
    print(f"Memory before: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

        elif col_type == object:
            if df[col].nunique(dropna=False)/len(df) < 0.5:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum()/1024**2
    print(f"Memory after : {end_mem:.2f} MB")
    print(f"Reduced by {(start_mem-end_mem)/start_mem*100:.1f}%")
    return df


## Load Dataset

**Kaggle paths** are enabled below. Uncomment the local paths if running on your PC.

In [ ]:
# Kaggle
train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e8/test.csv')
sample = pd.read_csv('/kaggle/input/playground-series-s6e8/sample_submission.csv')

# Local
# train = pd.read_csv('train.csv')
# test = pd.read_csv('test.csv')
# sample = pd.read_csv('sample_submission.csv')

train = reduce_memory(train)
test = reduce_memory(test)

print("Train:", train.shape)
print("Test :", test.shape)


## Dataset Overview

In [ ]:
display(train.head())
display(train.describe(include='all').T)
train.info()


## Missing Values

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
display(missing.to_frame("Missing"))


## Feature Lists

In [ ]:
FEATURES = [c for c in train.columns if c not in [ID, TARGET]]

CAT_FEATURES = train[FEATURES].select_dtypes(
    include=['object','category']
).columns.tolist()

NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

print("Categorical Features")
print(CAT_FEATURES)

print("\nNumerical Features")
print(NUM_FEATURES)


## Target Distribution

In [ ]:
print(train[TARGET].value_counts())
print()
print(train[TARGET].value_counts(normalize=True))

plt.figure(figsize=(5,4))
sns.countplot(x=train[TARGET])
plt.title("Target Distribution")
plt.show()


> ✅ Part 1 Complete.

Next notebook will contain a full professional EDA.